In [2]:
import pandas as pd
import numpy as np
import pickle

oobasic = pd.read_excel('DATAFILES/oonames.xlsx')
pickle.dump(oobasic, open('HOME_PICKLE_FILES/oonames.pkl','wb'))
instlist = pd.read_excel('DATAFILES/data_eco_bus.xlsx',sheet_name='listinstitutions')
pickle.dump(instlist, open('HOME_PICKLE_FILES/listinstitutions.pkl','wb'))

In [4]:
instlist = pd.read_pickle('HOME_PICKLE_FILES/listinstitutions.pkl')
oobasic = pd.read_pickle('HOME_PICKLE_FILES/oonames.pkl')
instlista = instlist[['ooidentifier','incitesname']]
incites = pd.read_pickle('HOME_PICKLE_FILES/inciteslist.pkl')

In [5]:
import glob, os
os.chdir("ECO_BUS_INST_FILES")
ifiles = sorted(glob.glob('*.txt'))

In [6]:
idatalist = []
inames = []

In [7]:
for file in ifiles:
    data = pd.read_csv(file, sep="\t",encoding='latin-1',low_memory=False)
    name = file.split('.')[0]
    inames.append(name)
    #print(name)
    if len(data.iloc[1, 0])>3:
        colnames = list(data.columns)
        colnames.append("final")
        data.columns = colnames[1:]
    data = data[['UT','SO','C1']]
    data = incites.merge(data, on='UT',how='left').dropna(subset=['SO'])
    data = data.sort_values('UT',ascending=True)
    data = data.reset_index(drop=True)
    ndata = data['C1'].map(str)
    ndata = ndata.apply(lambda x: pd.Series(str(x).split("]")))
    ndata = ndata.apply(lambda x: x.astype(str).str.upper())
    data = data[['UT','SO']]
    sandata = data.join(ndata)
    sandata = sandata.drop([0], axis=1)
    sdata = data.copy()
    oodata = data.copy()
    for k in range(1,len(ndata.columns)):
        df = pd.DataFrame(sandata[k].str.split(',').str[0].str.strip())
        oodata = pd.concat([oodata, pd.DataFrame(df[k])], axis=1)
        df = df.rename(columns={k:'oonames'})
        df = df.merge(oobasic, on='oonames',how='left')
        df = df.merge(instlista, on='ooidentifier',how='left')
        df = df.rename(columns={'incitesname':k})
        sdata = pd.concat([sdata, pd.DataFrame(df[k])], axis=1)
        ssdata = sdata.copy()
    todata = sdata[['UT','SO',1]]
    todata = todata.rename(columns={1:'incitesname'})
    antodata = todata.copy()
    for k in range(2,len(ndata.columns)):
        df = sdata[['UT','SO',k]].dropna()
        df = df.rename(columns={k:'incitesname'})
        todata = pd.concat([todata,df],ignore_index=True)
    todata = todata.dropna()
    todata = todata.rename(columns={'SO':'journal','incitesname':'institution'})
    todata = todata[['UT','journal','institution']]
    todata = todata.drop_duplicates(subset=['UT', 'journal'], keep='first')
    #mardata=  todata.groupby('UT')['journal'].count()
    #nuninst = pd.DataFrame(mardata)
    #nuninst = nuninst.reset_index()
    #nuninst = nuninst.rename(columns={'institution':'ninst'})
    #todata = todata.merge(mardata, on='UT',how='left')
    todata = todata.sort_values('UT',ascending=True)
    todata = todata.reset_index(drop=True)
    idatalist.append(todata)

In [8]:
os.chdir("..")
pickle.dump(idatalist, open('HOME_PICKLE_FILES/idatalist.pkl','wb'))
pickle.dump(ifiles, open('HOME_PICKLE_FILES/ifiles.pkl','wb'))
pickle.dump(inames, open('HOME_PICKLE_FILES/inames.pkl','wb'))

In [9]:
print('The end')

The end


In [10]:
len(idatalist)

600